In [ ]:
!pip install ultralytics pillow numpy --quiet
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"ultralytics version: {__import__('ultralytics').__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Edit these to match your Drive layout ──────────────────────────────
ASSETS_DIR  = '/content/drive/MyDrive/zoomy-pos/Complete Product Images - Transparent'
DATASET_DIR = '/content/dataset'
RUNS_DIR    = '/content/runs/detect/zoomy_yolo'
SAVE_DIR    = '/content/drive/MyDrive/zoomy-pos/yolo_model'

TRAIN_SCENES = 4000
VAL_SCENES   = 800
EPOCHS       = 150
BATCH_SIZE   = 16   # fits T4 GPU at 640×640
# ───────────────────────────────────────────────────────────────────────

import os, shutil
assert os.path.isdir(ASSETS_DIR), f"Assets not found: {ASSETS_DIR}"
print("Assets found:", os.listdir(ASSETS_DIR))

In [ ]:
# If you cloned the repo to /content/zoomy-pos:
!git clone https://github.com/gilvincent-work/zoomy-pos.git /content/zoomy-pos --depth 1 --quiet
import shutil
shutil.copy('/content/zoomy-pos/ml/generate_dataset.py', '/content/generate_dataset.py')
print("generate_dataset.py ready")

# If you prefer to upload manually: use Colab's file upload (Files panel on left)
# and skip this cell.

In [ ]:
!python3 /content/generate_dataset.py \
    --assets  "$ASSETS_DIR" \
    --output  "$DATASET_DIR" \
    --train   $TRAIN_SCENES \
    --val     $VAL_SCENES

# Verify
import os
n_train = len(os.listdir(f'{DATASET_DIR}/train/images'))
n_val   = len(os.listdir(f'{DATASET_DIR}/val/images'))
print(f"train images: {n_train}, val images: {n_val}")
assert n_train == TRAIN_SCENES
assert n_val   == VAL_SCENES

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.yaml')  # fresh nano architecture, no pretrained weights

results = model.train(
    data    = f'{DATASET_DIR}/data.yaml',
    epochs  = EPOCHS,
    imgsz   = 640,
    batch   = BATCH_SIZE,
    device  = 0,
    workers = 2,
    patience = 25,        # early-stop if no improvement for 25 epochs
    lr0     = 0.01,
    lrf     = 0.01,
    momentum= 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    mosaic  = 1.0,        # ultralytics mosaic augmentation
    degrees = 15.0,       # additional rotation augmentation
    translate = 0.1,
    scale   = 0.5,
    fliplr  = 0.5,
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    save    = True,
    project = '/content/runs/detect',
    name    = 'zoomy_yolo',
    exist_ok= True,
)

print(f"Best mAP@0.5 : {results.results_dict.get('metrics/mAP50(B)', 'N/A'):.3f}")
print(f"Best mAP@0.5:0.95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A'):.3f}")

In [ ]:
best = YOLO(f'{RUNS_DIR}/weights/best.pt')
metrics = best.val(data=f'{DATASET_DIR}/data.yaml', imgsz=640)

map50 = metrics.box.map50
print(f"mAP@0.5 = {map50:.3f}")

if map50 < 0.60:
    print("WARNING: mAP@0.5 below 0.60. Consider more epochs or more training scenes.")
    print("You can re-run Cell 5 with EPOCHS=200 or increase TRAIN_SCENES to 6000.")
else:
    print("Quality gate passed. Proceeding to export.")

In [ ]:
export_dir = best.export(format='tfjs', imgsz=640)
print(f"Exported to: {export_dir}")

# Verify output shape by listing files
import os
for f in os.listdir(export_dir):
    size = os.path.getsize(os.path.join(export_dir, f))
    print(f"  {f}  ({size/1024:.1f} KB)")

In [ ]:
import shutil, os

os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copytree(export_dir, f'{SAVE_DIR}/web_model', dirs_exist_ok=True)
print(f"Saved to {SAVE_DIR}/web_model/")
print()
print("Next steps to deploy:")
print("1. Download the contents of yolo_model/web_model/ from Drive")
print("2. Replace all files in public/ml-model/ with the downloaded files")
print("   (delete model.json, *.bin, keep labels.json)")
print("3. git add public/ml-model/ && git commit -m 'feat(model): deploy yolov8n detector'")
print("4. git push  →  Vercel redeploys automatically")